# Agente v140 en pendientes

Resultados de `slope_test.py` (100M @ 1.40 m/s). La pendiente se aplica inclinando la gravedad en un mundo plano; la política no percibe la pendiente. Caída = terminación por defecto del entorno.

Para regenerar los datos: `python pruebas_pendiente/slope_test.py --angles 0 5 10 15 20 25 30 40`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

path = Path('slope_results.json')
if not path.exists():
    path = Path('pruebas_pendiente/slope_results.json')
df = pd.DataFrame(json.loads(path.read_text()))
df['signed'] = df.apply(lambda r: r.angle if r['mode'] == 'up' else -r.angle, axis=1)
df = df.sort_values('signed')
df[['mode','angle','fall_frac','t_fall_median','t_fall_min','survive_s','vx_mean']]

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
cols = {'up': 'tab:red', 'down': 'tab:blue'}
names = {'up': 'ascenso', 'down': 'descenso'}
ref = df[df.angle == 0].iloc[0]
for mode, g in df.groupby('mode'):
    g = pd.concat([g, ref.to_frame().T]).drop_duplicates(['angle']).sort_values('angle') if mode == 'down' else g.sort_values('angle')
    ax[0].plot(g.angle, 100*g.fall_frac, 'o-', c=cols[mode], label=names[mode])
    ax[1].plot(g.angle, g.vx_mean, 'o-', c=cols[mode], label=names[mode])
    ax[2].plot(g.angle, g.survive_s, 'o-', c=cols[mode], label=names[mode])
ax[0].set_ylabel('robots caidos [%]'); ax[1].set_ylabel('velocidad media en x [m/s]'); ax[2].set_ylabel('tiempo medio sin caer [s]')
ax[1].axhline(0, c='k', lw=.6)
for a in ax:
    a.axvspan(5, 20, color='gray', alpha=.12); a.set_xlabel('inclinacion [grados]'); a.grid(alpha=.3); a.legend()
fig.suptitle('v140: banda gris = 5-20 grados (rango pedido)')
plt.tight_layout(); plt.show()

**Lectura:** sin caídas hasta 20° en ambos sentidos. El límite en ascenso está entre 20° y 25° (el robot pierde empuje y retrocede) y en descenso entre 25° y 30° (acelera demasiado).

## Videos
El suelo se ve inclinado (el mundo se gira para dejar la gravedad vertical). El robot avanza hacia la derecha; ascenso = sube. Se generan con `render_slopes.py`; el video se corta 0.5 s despues de una caida.

In [ ]:
from IPython.display import HTML, display
vdir = path.parent / 'videos'
cases = [f'{m}_{a}' for m in ('up','down') for a in (5,10,15,20)] + ['up_25','down_30']
cards = ''.join(
    f'<div style="display:inline-block;margin:4px"><b>{c.replace("up","ascenso").replace("down","descenso").replace("_"," ")}&deg;</b><br>'
    f'<video src="{(vdir/(c+".mp4")).as_posix()}" width="420" controls loop muted></video></div>'
    for c in cases if (vdir/(c+'.mp4')).exists())
display(HTML(cards))
